In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import _canoeValues as cv
import math
import time

#-------------------Inputs-------------------------------------------------------------
script = "Longitudinal Analysis_2 Paddler.ipynb"
input_file = "Station Information.csv"
outerHull_file = "Outer Hull.csv" # Will be used for hydrostatic force and waterline calculation
#--------------------------------------------------------------------------------------
script_dir = os.path.abspath('data') # absolute file path in Jupyter
file_path = os.path.join(script_dir, input_file) # file path for the station information

# Get the canoe's volume and mass from the station info
canoe_info = pd.read_csv(file_path, header=None)
statVol = canoe_info[1][1:]
statMass = canoe_info[2]

# Get the outer hull coordinates from the input file
file_path = os.path.join(script_dir, outerHull_file)
out_coor = pd.read_csv(file_path, header=None)
x_out = out_coor[0]
y_out = out_coor[1]
z_out = out_coor[2]

station_no = x_out.nunique() # number of stations
station = x_out.unique() # location of each station

waterline_file = "Waterline_2 Paddlers.csv"

file_path = os.path.join(script_dir, waterline_file) # file path for the station information
waterline_info = pd.read_csv(file_path, header=None)
waterline_z = [float(x) for x in waterline_info[1][1:]]
print(waterline_z)

In [ ]:
# Figure out how to import waterline points, then interpolate exterior hull points to get x
canoe_depth = max(abs(z_out)) 
vol_out = []
stat_waterline = [canoe_depth - abs(waterline) for waterline in waterline_z]
print(stat_waterline)
for k in range(0,station_no-1):
    # Station k coordinates (x,y,z)
    X1_value = station[k]
    Y1 = []
    Z1 = []
    
    # Station k+1 coordinates (x,y,z)
    X2_value = station[k+1]
    Y2 = []
    Z2 = []

    Y1_interp = []
    Y2_interp = []

    for i in range(0,len(x_out),1):
        # Get the points under the waterline
        if x_out[i] == X1_value:
            if canoe_depth - abs(z_out[i]) <= stat_waterline[k]:
                Y1.append(y_out[i])
                Z1.append(z_out[i])
                
                if canoe_depth - abs(z_out[i-1]) > stat_waterline[k]:

                    yInterpPoint = y_out[i-1] + ((y_out[i]-y_out[i-1]) / (z_out[i] - z_out[i-1])) * (stat_waterline[k] - canoe_depth - z_out[i-1])
                    
                    Y1_interp.append(yInterpPoint)
                            
                if canoe_depth - abs(z_out[i+1]) > stat_waterline[k]:
                    
                    yInterpPoint = y_out[i] + ((y_out[i+1]-y_out[i]) / (z_out[i+1] - z_out[i])) * (stat_waterline[k] - canoe_depth - z_out[i])
                    
                    Y1_interp.append(yInterpPoint)

        elif x_out[i] == X2_value:
            if canoe_depth - abs(z_out[i]) <= stat_waterline[k+1]:
                Y2.append(y_out[i])
                Z2.append(z_out[i])
                
                if canoe_depth - abs(z_out[i-1]) > stat_waterline[k+1]:

                    yInterpPoint = y_out[i-1] + ((y_out[i]-y_out[i-1]) / (z_out[i] - z_out[i-1])) * (stat_waterline[k+1] - canoe_depth - z_out[i-1])
                    
                    Y2_interp.append(yInterpPoint)
                            
                if canoe_depth - abs(z_out[i+1]) > stat_waterline[k+1]:
                    
                    yInterpPoint = y_out[i] + ((y_out[i+1]-y_out[i]) / (z_out[i+1] - z_out[i])) * (stat_waterline[k+1] - canoe_depth - z_out[i])
                    
                    Y2_interp.append(yInterpPoint)
                            
            if (len(Y1) == 0 and len(Y2) == 0) or (len(Y1_interp) == 0 and len(Y2_interp) == 0):
                # Station is out of the water
                vol_out.append(0)
                continue
            elif len(Y1) == 0 or len(Y2) == 0 or (len(Y1_interp) == 0 or len(Y2_interp) == 0):
                # Interpolate the waterline for the submerged station
                if len(Y1) == 0:
                    Y2 = [Y2_interp[0]] + Y2 + [Y2_interp[-1]]
                    Z2 = [stat_waterline[k+1] - canoe_depth] + Z2 + [stat_waterline[k+1] - canoe_depth]
                    
                    Y1 = [0]
                    Z1 = [stat_waterline[k]]
                else:
                    Y1 = [Y1_interp[0]] + Y1 + [Y1_interp[-1]]
                    Z1 = [stat_waterline[k] - canoe_depth] + Z1 + [stat_waterline[k] - canoe_depth]
                    
                    Y2 = [0]
                    Z2 = [stat_waterline[k+1]]
            else:
                print(Y1_interp)
                Y1 = [Y1_interp[0]] + Y1 + [Y1_interp[-1]]
                Y2 = [Y2_interp[0]] + Y2 + [Y2_interp[-1]]
                
                Z1 = [stat_waterline[k] - canoe_depth] + Z1 + [stat_waterline[k] - canoe_depth]
                Z2 = [stat_waterline[k+1] - canoe_depth] + Z2 + [stat_waterline[k+1] - canoe_depth]



In [ ]:
# Moment of inertia calculation, figure out how this works
I = 0
for i in range(0,len(Y1_out)-1,1):
    z_i = ((Z1_out[i]+Z1_out[i+1]+Z1_in[i]+Z1_in[i+1])/4 + (Z2_out[i]+Z2_out[i+1]+Z2_in[i]+Z2_in[i+1])/4) /2
    y = (abs(Y1_in[i] - Y1_out[i]) + abs(Y1_in[i+1]-Y1_out[i+1]) + abs(Y2_in[i] - Y2_out[i]) + abs(Y2_in[i+1]-Y2_out[i+1]))/4
    z = (abs(Z1_out[i+1]-Z1_out[i]) + abs(Z1_in[i+1]-Z1_in[i]) + abs(Z2_out[i+1]-Z2_out[i]) + abs(Z2_in[i+1]-Z2_in[i]))/4
    I_i = (1/12*y*z**3 + y*z*(abs(z_i) - y_bar_top)**2)/10**(12) # [m^4]
    I = I + I_i